# Bone Fracture Detection — Phase 4-6: Evaluate & Visualise

**Pre-requisite:** Phase 3 training ran for 64 epochs (12h Kaggle limit). `best.pt` is available from the previous run output.

**What this notebook does (fast, ~5 min total GPU time):**
1. Rebuild merged dataset (CPU only)
2. Load `best.pt` from previous output
3. Phase 4 — Evaluate on test set
4. Phase 5 — Training curves, confusion matrix, PR/F1 curves, sample predictions
5. Phase 6 — Final summary

**Kaggle setup:** Add these as Input datasets:
- `archive` (BoneFractureYolo8)
- `fracatlasdataset` (FracAtlas)
- `graz-yolo-ready-v11` (GRAZPEDWRI-DX)
- Your previous notebook's **Output** (contains `runs/fracture_v3/`)

## Setup — Environment & Find best.pt

In [ ]:
import subprocess, torch, os, shutil, random, yaml, csv, zipfile, glob
from pathlib import Path
from collections import Counter

subprocess.run(["pip", "install", "-q", "ultralytics>=8.2.0"], check=True)

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Enable GPU in Kaggle settings.")

num_gpus = torch.cuda.device_count()
for i in range(num_gpus):
    p = torch.cuda.get_device_properties(i)
    fg = torch.cuda.mem_get_info(i)[0] / 1e9
    print(f"  GPU {i}: {p.name} | {fg:.1f} GB free")

DEVICE       = "0,1" if num_gpus >= 2 else "0"
KAGGLE_INPUT = Path("/kaggle/input")
WORKING_DIR  = Path("/kaggle/working")

# ── Find best.pt from previous run output ──
BEST_PT = None
RUN_DIR = None  # training run dir (for plots)

# Search strategy:
#  1. Previous notebook output added as input dataset
#  2. Directly in working dir (if resuming same session)
#  3. Download from GitHub releases (fallback)

search_paths = [
    # Previous output added as dataset (Kaggle flattens or keeps structure)
    *sorted(KAGGLE_INPUT.rglob("best.pt")),
    WORKING_DIR / "runs" / "fracture_v3" / "weights" / "best.pt",
]

for p in search_paths:
    if p.exists():
        BEST_PT = p
        print(f"[OK] Found best.pt: {BEST_PT}")
        break

# Also look for best.zip and extract
if BEST_PT is None:
    for zp in sorted(KAGGLE_INPUT.rglob("best.zip")):
        print(f"[INFO] Found best.zip: {zp}, extracting...")
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(WORKING_DIR / "_weights")
        for pt in (WORKING_DIR / "_weights").rglob("*.pt"):
            BEST_PT = pt
            break
        if BEST_PT:
            print(f"[OK] Extracted best.pt: {BEST_PT}")
            break

# Fallback: download from GitHub
if BEST_PT is None:
    print("[INFO] Downloading best.pt from GitHub releases...")
    import urllib.request
    url = "https://github.com/Atishay2204/bone-fracture-detection/releases/download/v1.0/best.zip"
    dl_path = WORKING_DIR / "best_dl.zip"
    urllib.request.urlretrieve(url, str(dl_path))
    with zipfile.ZipFile(dl_path, "r") as zf:
        zf.extractall(WORKING_DIR / "_weights")
    for pt in (WORKING_DIR / "_weights").rglob("*.pt"):
        BEST_PT = pt
        break
    if BEST_PT is None and dl_path.exists():
        BEST_PT = WORKING_DIR / "best.pt"
        os.rename(dl_path, BEST_PT)
    print(f"[OK] Downloaded best.pt: {BEST_PT}")

assert BEST_PT and BEST_PT.exists(), "ERROR: Cannot find best.pt anywhere!"

# Find training run dir (for results.png, confusion matrix, etc.)
for rd in sorted(KAGGLE_INPUT.rglob("results.csv")):
    RUN_DIR = rd.parent
    break
if RUN_DIR is None:
    rd2 = WORKING_DIR / "runs" / "fracture_v3"
    if rd2.exists():
        RUN_DIR = rd2

print(f"Run dir (for plots): {RUN_DIR}")
print("Setup complete.")

## Rebuild Merged Dataset (CPU only — ~1 min)

In [ ]:
# ── Find the 3 source datasets ──
SKIP_WALK = {
    "images","labels","train","valid","val","test",
    "images_part1","images_part2","images_part3","images_part4",
    "Fractured","Non-Fractured","yolov5","YOLO","yolo",
    "supervisely","wrist","pascalvoc","PASCAL VOC","VGG JSON","COCO",
    "notebooks","Augmentation","Splits","Fracture Split",
    "bone fracture detection.v4-v4.yolov8",
}

def fast_find(keyword, need_dir=None, need_file=None):
    kw = keyword.lower()
    found = []
    def _w(p, d):
        if d > 7: return
        try:
            for c in p.iterdir():
                if not c.is_dir() or c.name in SKIP_WALK: continue
                if kw in c.name.lower():
                    ok = True
                    if need_dir  and not (c/need_dir).is_dir():   ok = False
                    if need_file and not (c/need_file).is_file():
                        ok = any((s/need_file).is_file()
                                 for s in c.iterdir()
                                 if s.is_dir() and s.name not in SKIP_WALK)
                    if ok: found.append(c)
                _w(c, d+1)
        except PermissionError: pass
    _w(KAGGLE_INPUT, 0)
    return sorted(found, key=lambda x: len(x.parts))[0] if found else None

CANDIDATES = {
    "BONE8": [
        KAGGLE_INPUT/"datasets"/"atishayjain2204"/"archive"/"BoneFractureYolo8",
        KAGGLE_INPUT/"archive"/"BoneFractureYolo8",
    ],
    "FA": [
        KAGGLE_INPUT/"datasets"/"atishayjain2204"/"fracatlasdataset"/"FracAtlas",
        KAGGLE_INPUT/"fracatlasdataset"/"FracAtlas",
        KAGGLE_INPUT/"fracAtlasDataSet"/"FracAtlas",
    ],
    "GRAZ": [
        KAGGLE_INPUT/"datasets"/"avizith"/"graz-yolo-ready-v11",
        KAGGLE_INPUT/"graz-yolo-ready-v11",
    ],
}

def pick(key):
    for p in CANDIDATES[key]:
        if p.exists(): return p
    return None

BONE8_DIR = pick("BONE8") or fast_find("BoneFractureYolo8", need_dir="train")
FA_DIR    = pick("FA")    or fast_find("FracAtlas", need_dir="Annotations")
GRAZ_DIR  = pick("GRAZ")  or fast_find("graz-yolo-ready", need_file="data.yaml")
if GRAZ_DIR is None:
    GRAZ_DIR = fast_find("graz", need_file="data.yaml")

for label, path in [("BoneFractureYolo8", BONE8_DIR),
                    ("FracAtlas", FA_DIR),
                    ("graz-yolo-ready", GRAZ_DIR)]:
    ok = path is not None and path.exists()
    print(f"  [{'OK' if ok else 'MISSING'}] {label:20s}: {path}")
    if not ok:
        raise FileNotFoundError(f"{label} not found! Add it as Kaggle input dataset.")
print("All 3 datasets found.")

In [ ]:
# ── 9-class unified schema & merge ──
UNIFIED_CLASSES = [
    "elbow_fracture", "finger_fracture", "forearm_fracture",
    "humerus_fracture", "shoulder_fracture", "wrist_fracture",
    "hand_fracture", "hip_fracture", "leg_fracture",
]
NC = len(UNIFIED_CLASSES)
BONE8_REMAP = {0:0, 1:1, 2:2, 3:3, 4:3, 5:4, 6:5}
FA_BODYPART = {"Hand":6, "Hip":7, "Leg":8, "Shoulder":4}

GRAZ_YAML     = yaml.safe_load((GRAZ_DIR / "data.yaml").read_text())
graz_names    = GRAZ_YAML.get("names", [])
GRAZ_FRAC_CLS = next((i for i, n in enumerate(graz_names) if "fracture" in str(n).lower()), 3)
GRAZ_TGT_CLS  = 5  # -> wrist_fracture

MERGED_DIR  = WORKING_DIR / "merged_dataset"
MERGED_YAML = WORKING_DIR / "merged_data.yaml"
random.seed(42)

for sp in ["train", "val", "test"]:
    (MERGED_DIR / sp / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / sp / "labels").mkdir(parents=True, exist_ok=True)

counter = Counter()

def get_images(d):
    imgs = []
    for ext in ["*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG"]:
        imgs.extend(d.glob(ext))
    return imgs

def add_sample(img, lines, split, prefix):
    stem = f"{prefix}_{img.stem}"
    shutil.copy2(img, MERGED_DIR / split / "images" / f"{stem}{img.suffix}")
    (MERGED_DIR / split / "labels" / f"{stem}.txt").write_text("\n".join(lines))
    for line in lines:
        counter[UNIFIED_CLASSES[int(line.split()[0])]] += 1

def remap(lbl, remap_dict):
    if not lbl or not lbl.exists(): return []
    out = []
    for line in lbl.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts and int(parts[0]) in remap_dict:
            parts[0] = str(remap_dict[int(parts[0])])
            out.append(" ".join(parts))
    return out

def keep_cls(lbl, keep, new_cls):
    if not lbl or not lbl.exists(): return []
    out = []
    for line in lbl.read_text().strip().splitlines():
        parts = line.strip().split()
        if parts and int(parts[0]) == keep:
            parts[0] = str(new_cls)
            out.append(" ".join(parts))
    return out

# --- Dataset 1: BoneFractureYolo8 ---
print("Merging BoneFractureYolo8...")
for src_sp in ["train", "valid", "test"]:
    dst_sp = "val" if src_sp == "valid" else src_sp
    img_dir = BONE8_DIR / src_sp / "images"
    lbl_dir = BONE8_DIR / src_sp / "labels"
    if not img_dir.exists(): continue
    for img in get_images(img_dir):
        add_sample(img, remap(lbl_dir / f"{img.stem}.txt", BONE8_REMAP), dst_sp, "b8")

# --- Dataset 2: FracAtlas ---
print("Merging FracAtlas...")
YOLO_DIR = None
for cand in [FA_DIR/"Annotations"/"YOLO", FA_DIR/"Annotations"/"yolo", FA_DIR/"annotations"/"YOLO"]:
    if cand.exists() and any(cand.glob("*.txt")):
        YOLO_DIR = cand; break
if YOLO_DIR is None:
    for d in FA_DIR.rglob("*"):
        if d.is_dir() and d.name.upper() == "YOLO" and any(d.glob("*.txt")):
            YOLO_DIR = d; break
assert YOLO_DIR, f"YOLO labels not found in {FA_DIR}"

fractured_root = FA_DIR / "images" / "Fractured"
assert fractured_root.exists(), f"images/Fractured/ not found in {FA_DIR}"

samples = []
body_part_dirs = [d for d in fractured_root.iterdir() if d.is_dir() and d.name in FA_BODYPART]
if body_part_dirs:
    for bp_dir in body_part_dirs:
        cls = FA_BODYPART[bp_dir.name]
        for img in get_images(bp_dir):
            samples.append((img, cls, YOLO_DIR / f"{img.stem}.txt"))
else:
    for img in get_images(fractured_root):
        samples.append((img, 4, YOLO_DIR / f"{img.stem}.txt"))

random.shuffle(samples)
n = len(samples); nt = int(n*0.75); nv = int(n*0.15)
for sp, subset in [("train", samples[:nt]), ("val", samples[nt:nt+nv]), ("test", samples[nt+nv:])]:
    for img, cls, lbl in subset:
        add_sample(img, remap(lbl, {0: cls}), sp, "fa")

# --- Dataset 3: GRAZPEDWRI-DX ---
print("Merging GRAZPEDWRI-DX...")
img_base = GRAZ_DIR / "images"
lbl_base = GRAZ_DIR / "labels"
split_subs = [s for s in ["train","valid","val","test"] if (img_base/s).is_dir()]
if split_subs:
    for src_sp in split_subs:
        dst_sp = "val" if src_sp in ("valid","val") else src_sp
        for img in get_images(img_base / src_sp):
            add_sample(img, keep_cls(lbl_base/src_sp/f"{img.stem}.txt", GRAZ_FRAC_CLS, GRAZ_TGT_CLS), dst_sp, "gz")
else:
    all_gz = get_images(img_base)
    random.shuffle(all_gz)
    n = len(all_gz); nt = int(n*0.75); nv = int(n*0.15)
    for sp, subset in [("train",all_gz[:nt]),("val",all_gz[nt:nt+nv]),("test",all_gz[nt+nv:])]:
        for img in subset:
            add_sample(img, keep_cls(lbl_base/f"{img.stem}.txt", GRAZ_FRAC_CLS, GRAZ_TGT_CLS), sp, "gz")

# --- Write YAML & summary ---
cfg = {
    "path": str(MERGED_DIR), "train": "train/images",
    "val": "val/images", "test": "test/images",
    "nc": NC, "names": UNIFIED_CLASSES,
}
MERGED_YAML.write_text(yaml.dump(cfg, default_flow_style=False, sort_keys=False))

print("\n" + "="*60)
print("MERGED DATASET SUMMARY")
grand = 0
for sp in ["train", "val", "test"]:
    n = len(list((MERGED_DIR/sp/"images").glob("*")))
    grand += n
    print(f"  {sp:5s}: {n:,} images")
print(f"  TOTAL: {grand:,}")
print(f"\nAnnotations per class:")
mx = max(counter.values(), default=1)
for cls in UNIFIED_CLASSES:
    cnt = counter.get(cls, 0)
    bar = chr(9608) * int(35 * cnt / mx)
    print(f"  {cls:20s}: {cnt:6,}  {bar}")
print(f"Total annotations: {sum(counter.values()):,}")
print("Merge complete!")

## Phase 4 — Evaluate on Test Set

In [ ]:
from ultralytics import YOLO

IMGSZ = 800
print(f"Loading model: {BEST_PT}")
best = YOLO(str(BEST_PT))

(WORKING_DIR / "outputs").mkdir(exist_ok=True)

metrics = best.val(
    data=str(MERGED_YAML), split="test", device=DEVICE,
    imgsz=IMGSZ, batch=8, plots=True, save_json=True,
    project=str(WORKING_DIR / "outputs"), name="test_eval",
)

print("=" * 60)
print(f"mAP@50    : {metrics.box.map50:.4f}")
print(f"mAP@50-95 : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print()
print("Per-class AP@50:")
for cls, ap in zip(UNIFIED_CLASSES, metrics.box.ap50):
    flag = "  << low" if ap < 0.3 else ""
    print(f"  {cls:20s}: {ap:.4f}{flag}")

## Phase 5 — Plots & Sample Predictions

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

OUT_DIR = WORKING_DIR / "outputs"
OUT_DIR.mkdir(exist_ok=True)

# Try to find training plots from the previous run
# Check: previous output (added as input), working dir run, or eval output
plot_sources = []
if RUN_DIR and RUN_DIR.exists():
    plot_sources.append(RUN_DIR)
plot_sources.append(OUT_DIR / "test_eval")

for fname, title in [
    ("results.png",                     "Training Curves"),
    ("confusion_matrix_normalized.png", "Confusion Matrix (Normalised)"),
    ("confusion_matrix.png",            "Confusion Matrix"),
    ("PR_curve.png",                    "Precision-Recall Curve"),
    ("F1_curve.png",                    "F1 Curve"),
    ("P_curve.png",                     "Precision Curve"),
    ("R_curve.png",                     "Recall Curve"),
    ("labels.jpg",                      "Label Distribution"),
]:
    found = False
    for src in plot_sources:
        p = src / fname
        if p.exists():
            plt.figure(figsize=(14, 8))
            plt.imshow(mpimg.imread(str(p)))
            plt.axis("off")
            plt.title(title, fontsize=14)
            plt.tight_layout()
            plt.savefig(OUT_DIR / fname, dpi=120, bbox_inches="tight")
            plt.show()
            found = True
            break
    if not found:
        print(f"Not found: {fname} (training plots require the previous run's output as input dataset)")

In [ ]:
import random as _r
import matplotlib.pyplot as plt
from pathlib import Path

test_imgs = list((MERGED_DIR / "test" / "images").glob("*"))
sample    = _r.sample(test_imgs, k=min(6, len(test_imgs)))
preds     = best.predict(
    source=[str(p) for p in sample],
    imgsz=IMGSZ, device=DEVICE, conf=0.25, iou=0.45, verbose=False,
)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, r in zip(axes.flatten(), preds):
    ax.imshow(r.plot()[..., ::-1])
    ax.axis("off")
    ds = {"b8": "BoneFrac", "fa": "FracAtlas", "gz": "GRAZ"}.get(
        Path(r.path).name[:2], "?")
    ax.set_title(f"[{ds}] {len(r.boxes) if r.boxes else 0} det(s)", fontsize=9)
plt.suptitle("Sample Predictions (conf>=0.25)", fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / "predictions.png", dpi=100, bbox_inches="tight")
plt.show()

## Phase 6 — Final Summary & Save Outputs

In [ ]:
import shutil

print("=" * 60)
print("EVALUATION COMPLETE")
print(f"Model       : best.pt (64 epochs, YOLOv8l, AdamW, imgsz=800)")
print(f"mAP@50      : {metrics.box.map50:.4f}")
print(f"mAP@50-95   : {metrics.box.map:.4f}")
print(f"Precision   : {metrics.box.mp:.4f}")
print(f"Recall      : {metrics.box.mr:.4f}")
print()
print("Per-class AP@50:")
for cls, ap in zip(UNIFIED_CLASSES, metrics.box.ap50):
    print(f"  {cls:20s}: {ap:.4f}")
print()
print(f"best.pt     : {BEST_PT}")
print(f"YAML        : {MERGED_YAML}")
print(f"Eval output : {OUT_DIR / 'test_eval'}")

# Copy best.pt to outputs for easy download
shutil.copy2(BEST_PT, OUT_DIR / "best.pt")
print(f"\nbest.pt copied to: {OUT_DIR / 'best.pt'}")
print("All outputs saved to /kaggle/working/outputs/")

## Next Steps (Optional)

If you want to **resume training** for more epochs in a future session:

```python
# In a new Kaggle notebook, add the previous output as input dataset
# Then find last.pt and resume:
from ultralytics import YOLO
model = YOLO("/kaggle/input/<your-output-dataset>/runs/fracture_v3/weights/last.pt")
results = model.train(resume=True)
```

**Tip:** If `last.pt` isn't available, you can fine-tune from `best.pt` with a lower learning rate:
```python
model = YOLO("best.pt")
results = model.train(
    data="merged_data.yaml", epochs=100, imgsz=800, batch=16,
    optimizer="AdamW", lr0=0.0002, patience=30,
    # ... same augmentations as before
)
```